In [ ]:
import nilearn
import numpy as np
import nibabel as nib
from sklearn.cluster import KMeans
from nilearn import plotting as nplt
from sklearn.metrics import pairwise_distances
from pathlib import Path
from sklearn.metrics import davies_bouldin_score, silhouette_score
import pandas as pd
import matplotlib.pyplot as plt
from neuromaps.parcellate import Parcellater
from nilearn.datasets import fetch_atlas_schaefer_2018
from tqdm import tqdm
from nilearn.image import resample_to_img

random_state = 42

# Introduction
VMPFC parcellation using neurotransmitter receptor / transporter PET maps from the Hansen Receptors Atlas (Hansen et al., 2022; 19 group-average PET maps spanning nine neurotransmitter systems aggregated from >1,200 healthy adults).

Procedure: each PET volume is smoothed (FWHM = 4 mm), then for every VMPFC voxel a feature vector is built as its cortical similarity profile to the rest of the brain summarised by the Schaefer-2018 1000-region atlas (via neuromaps.Parcellater). Voxel-by-voxel cross-correlation of these profiles is then clustered with K-means (K = 2–6, n_init = 1000, random_state = 42), matching the procedure used for the other modalities in this study.


Hansen, J. Y., Shafiei, G., Markello, R. D., Smart, K., Cox, S. M., Nørgaard, M., ... & Misic, B. (2022). Mapping neurotransmitter systems to the structural and functional organization of the human neocortex. _Nature neuroscience_ _25_(11), 1569-1581.

# 2. Settings

In [ ]:
DATASET_FOLDER = Path("/home/guoqiu/NAS/NAS4/guoqiu/Database/Neurotransmitter")
FILENAME_PATTERN = '*.nii'

DATA_FOLDER = Path('../data')
RESULT_FOLDER = Path('../results')

In [ ]:
ROI_IMAGE = nib.load('../data/VMPFC_3mm.nii')
ROI_MASK = ROI_IMAGE.get_fdata() > 0
unROI_IMAGE = nib.load('../data/GM_mask_3mm.nii')
unROI_MASK = unROI_IMAGE.get_fdata() > 0

N_CLUSTERS_LIST = list(range(2, 7))
FWHM = 4

In [ ]:
schaefer = fetch_atlas_schaefer_2018(n_rois=1000)
parcellater = Parcellater(schaefer['maps'], 'MNI152').fit()

In [ ]:
# connectivity was defined as similarity between subjects
ROI_data, unROI_data = list(), list()
subject_file_list = list(DATASET_FOLDER.rglob(FILENAME_PATTERN))
for subject_file in tqdm(subject_file_list, desc='Load data'):
    subject_img = nilearn.image.smooth_img(subject_file, fwhm=FWHM)
    subject_data = subject_img.get_fdata()
    this_ROI_data = subject_data[ROI_MASK].flatten()
    if np.isnan(this_unROI_data).any(): continue
    this_ROI_data = np.nan_to_num(this_ROI_data)

    ROI_data.append(this_ROI_data)

    this_unROI_data = parcellater.transform(subject_file, 'MNI152', True).flatten()
    this_unROI_data = np.nan_to_num(this_unROI_data)
    unROI_data.append(this_unROI_data)
ROI_data, unROI_data = np.array(ROI_data).T, np.array(unROI_data).T
print(f'ROI: (n voxels, n subjects) = {ROI_data.shape}')
print(f'unROI: (n voxels, n subjects) = {unROI_data.shape}')

connectivity = pairwise_distances(ROI_data, unROI_data, metric='correlation').T
print(f'connectivity: (n unROI voxels, n ROI voxels) = {connectivity.shape}')
assert not np.isnan(connectivity).any(), "There are NaNs in connectivity!"

cross_correlation = np.corrcoef(connectivity, rowvar=False)
print(f'cross_correlation: (n ROI voxels, n ROI voxels) = {cross_correlation.shape}')
assert not np.isnan(cross_correlation).any(), "There are NaNs in cross_correlation!"

In [ ]:
label_dict = {
    n_clusters:
        KMeans(
            n_clusters=n_clusters,
            init='random',
            n_init=1000,
            random_state=random_state,
        ).fit_predict(cross_correlation) + 1
    for n_clusters in N_CLUSTERS_LIST
}

# Validity measure of clusters - higher is better
SS_scores = [silhouette_score(cross_correlation, label) for label in label_dict.values()]
DBI_scores = [davies_bouldin_score(cross_correlation, label) for label in label_dict.values()]
scores_df = pd.DataFrame(dict(silhouette_scores=SS_scores, davies_bouldin_scores=DBI_scores))
scores_df.index = N_CLUSTERS_LIST
scores_df.index.name = 'n_clusters'
scores_df.to_csv( RESULT_FOLDER / 'internal_validity/Neurotransmitter.csv')

# fast plot for the scores
line1, = plt.plot(N_CLUSTERS_LIST, SS_scores, 'b-*', )
plt.twinx()
line2, = plt.plot(N_CLUSTERS_LIST, DBI_scores, 'ro-', )
plt.legend([line1, line2], ['Silhouette Score', 'Davies Bouldin Score'])

In [ ]:
for n_clusters in N_CLUSTERS_LIST:
    label_img_data = np.zeros_like(ROI_MASK, dtype=np.int8)
    label_img_data[ROI_MASK] = label_dict[n_clusters]
    label_img = nib.nifti1.Nifti1Image(label_img_data, header=ROI_IMAGE.header, affine=ROI_IMAGE.affine, )
    nib.save(label_img, str(RESULT_FOLDER / f'nii/Neurotransmitter_K{n_clusters}_3mm.nii.gz'))
    # resample to 2mm for later comparison
    resampled_img = resample_to_img(label_img, DATA_FOLDER / 'masks/VMPFC_2mm.nii', interpolation='nearest')
    resampled_img.to_filename( str(RESULT_FOLDER / f'nii/Neurotransmitter_K{n_clusters}.nii.gz'))
    # fast plot for labels
    nplt.plot_stat_map(label_img, title=f'N={n_clusters}', cmap='tab10', draw_cross=False, colorbar=False, )
    nplt.plot_stat_map(resampled_img, title=f'N={n_clusters}', cmap='tab10', draw_cross=False, colorbar=False, )